# Chargées les 9 tables d'Olist dans un dictionnaire

In [ ]:
# téléchargement du jeu de données de Olist sur Kaggle et stocker dans path
import kagglehub
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [ ]:
# import de la bibliothèque pandas pour charger et analyser les données
import pandas as pd
# import de la bibliothèque os pour intéragir avec les fichiers
import os
import numpy as np
import re

os.listdir(path)   # lister les 9 fichiers

['olist_customers_dataset.csv',
 'olist_sellers_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_products_dataset.csv',
 'olist_geolocation_dataset.csv',
 'product_category_name_translation.csv',
 'olist_orders_dataset.csv',
 'olist_order_payments_dataset.csv']

In [ ]:
fichiers = os.listdir(path)

mes_tables = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_products_dataset.csv',
    'product_category_name_translation.csv'
]

tables = {}   # dictionnaire vide
for f in fichiers:
    if f in mes_tables :
      tables[f] = pd.read_csv(os.path.join(path, f))

tables.keys() # Vérification

dict_keys(['olist_customers_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv'])

# CLEAN

## Orders

In [ ]:
orders = tables['olist_orders_dataset.csv']

In [ ]:
# Forme : nombre d'enregistrements / attributs
orders.shape

(99441, 8)

In [ ]:
# Liste des attributs
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

* order_id : identifiant unique de la commande

* customer_id : identifiant du client pour cette commande

* order_status : état de la commande (livré, expédié, annulé, indisponible, facturé, en cours de traitement, créé, approuvé)

* order_purchase_timestamp : quand le client a passé la commande

* order_approved_at : quand le paiement a été approuvé/validé

* order_delivered_carrier_date : quand le colis a été remis au transporteur

* order_delivered_customer_date : quand le client a reçu le colis

* order_estimated_delivery_date : estimation de la date de livraison au client au moment de l'achat

In [ ]:
# Aperçu des 5 premières lignes
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
# Vérification du type de chaque attributs
orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


In [ ]:
# Valeurs manquantes par colonne
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [ ]:
# Unicité (doublons ?)
orders['order_id'].duplicated().sum()

np.int64(0)

In [ ]:
# Unicité (doublons ?)
orders['customer_id'].duplicated().sum()

np.int64(0)

In [ ]:
# Répartition des statuts de commande
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [ ]:
# changement du type objet au type date
colonnes_dates = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in colonnes_dates:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [ ]:
# Vérification du type de chaque attributs de date
orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


## Customers

In [ ]:
customers = tables['olist_customers_dataset.csv']

In [ ]:
customers.shape

(99441, 5)

In [ ]:
customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

* customer_id : identifiant client par commande

* customer_unique_id : identifiant unique du client

* customer_zip_code_prefix : préfixe du code postal

* customer_city : ville du client

* customer_state : état brésilien

In [ ]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# Vérification du type de chaque attributs
customers.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


In [ ]:
# Valeurs manquantes par colonne
customers.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


In [ ]:
# Unicité (doublons ?)
customers['customer_id'].duplicated().sum()

np.int64(0)

In [ ]:
# Unicité (doublons ?)
customers['customer_unique_id'].duplicated().sum()

np.int64(3345)

In [ ]:
customers['customer_city'].describe()

,customer_city
count,99441
unique,4119
top,sao paulo
freq,15540


In [ ]:
customers['customer_state'].describe()

,customer_state
count,99441
unique,27
top,SP
freq,41746


## Items

In [ ]:
items = tables['olist_order_items_dataset.csv']

In [ ]:
items.shape

(112650, 7)

In [ ]:
items.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

* order_id : identifiant unique de la commande

* order_item_id : compteur, numéro de l'article dans la commande(1, 2, 3)

* product_id : identifiant produit

* seller_id : identifiant vendeur

* shipping_limit_date : date limite d'éxpédition

* price : prix de l'article

* freight_value : frais de port de l'article

In [ ]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
items.dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64


In [ ]:
items.isnull().sum()

,0
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


In [ ]:
# Unicité (doublons ?)
items.duplicated().sum()

np.int64(0)

In [ ]:
# Unicité (doublons ?)
items.duplicated(subset=['order_id']).sum()

np.int64(13984)

In [ ]:
# Unicité (doublons ?)
items.duplicated(subset=['order_id', 'order_item_id']).sum()

np.int64(0)

In [ ]:
items[['price', 'freight_value']].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


## Payements

In [ ]:
payements = tables['olist_order_payments_dataset.csv']

In [ ]:
payements.shape

(103886, 5)

In [ ]:
payements.columns.tolist()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

* order_id : identifiant de la commande

* payment_sequential : nombre de méthode de paiement (de 1 à  29 méthode de paiement)

* payment_type : type de paiement(carte de crédit, boleto bancário(titre de paiement avec code-barres), bon, carte de débit)

* payment_installments : nombre de mensualités (de 1 à 23 mensualités)

* payment_value : valeur de l'achat

In [ ]:
payements.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
payements.dtypes

,0
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64


In [ ]:
payements.isnull().sum()

,0
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


In [ ]:
payements.duplicated().sum()

np.int64(0)

In [ ]:
payements.duplicated(subset=['order_id']).sum()

np.int64(4446)

In [ ]:
payements.duplicated(subset=['order_id', 'payment_sequential']).sum()

np.int64(0)

In [ ]:
payements['payment_sequential'].value_counts()

,count
payment_sequential,
1,99360
2,3039
3,581
4,278
5,170
6,118
7,82
8,54
9,43


In [ ]:
payements['payment_type'].value_counts()

,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [ ]:
payements['payment_installments'].value_counts()

,count
payment_installments,
1,52546
2,12413
3,10461
4,7098
10,5328
5,5239
8,4268
6,3920
7,1626


In [ ]:
payements[['payment_value']].describe()

,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000


In [ ]:
(payements['payment_value'] == 0).sum()

np.int64(9)

## Products

In [ ]:
products = tables['olist_products_dataset.csv']

In [ ]:
products.shape

(32951, 9)

In [ ]:
products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

* product_id : identifiant du produit

* product_category_name : catégorie(73) du produit

* product_name_lenght : longueur du nom du produit

* product_description_lenght : longueur de description du produit

* product_photos_qty : quantité de photos du produit

* product_weight_g : poids(gr) du produit

* product_length_cm : longueur(cm) du produit

* product_height_cm : hauteur(cm) du produit

* product_width_cm : largeur(cm) du produit

In [ ]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [ ]:
products.dtypes

,0
product_id,object
product_category_name,object
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64


In [ ]:
products.isnull().sum()

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [ ]:
products.duplicated().sum()

np.int64(0)

In [ ]:
products.duplicated(subset=['product_id']).sum()

np.int64(0)

In [ ]:
products['product_category_name'].describe()

,product_category_name
count,32341
unique,73
top,cama_mesa_banho
freq,3029


## Translation

In [ ]:
translation = tables['product_category_name_translation.csv']

In [ ]:
translation.shape

(71, 2)

In [ ]:
translation.columns.tolist()

['product_category_name', 'product_category_name_english']

* product_category_name : catégorie de produit

* product_category_name_english : traduction anglaise des catégorie

In [ ]:
translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [ ]:
translation.isnull().sum()

,0
product_category_name,0
product_category_name_english,0


In [ ]:
translation.duplicated().sum()

np.int64(0)

# Téléchargement

In [ ]:
# Création des fichiers CSV nettoyés
orders.to_csv('orders_clean.csv', index=False)
customers.to_csv('customers_clean.csv', index=False)
items.to_csv('items_clean.csv', index=False)
payements.to_csv('payments_clean.csv', index=False)
products.to_csv('products_clean.csv', index=False)
translation.to_csv('translation_clean.csv', index=False)

In [ ]:
# Téléchargements
from google.colab import files

files.download('orders_clean.csv')
files.download('customers_clean.csv')
files.download('items_clean.csv')
files.download('payments_clean.csv')
files.download('products_clean.csv')
files.download('translation_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>